In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
from memoria_icm.datos import *
from memoria_icm.funciones import *
from memoria_icm.graficos import *

Este cuadernillo analiza la existencia de diferencias significativas entre los grupos de la misma especie. EXPOSICIÓN.

In [2]:
df, df_resumen = cargar_datos()
df

Datos cargados. Trackeo: (44324, 28), Resumen: (192, 42)


,index,id,t,x,y,id_ubicacion,especie,tratamiento,salinidad,nombre_archivo,...,a,r,dr,theta_r,v_angular,a_angular,theta_v,curvatura,duracion_video_recortada,nivel_salinidad
0,0,0,0.000000,1.976667,-0.624416,1,CC,E,35-N,Video_001,...,2.047467,0.000000,0.098320,0.000000,-12.040794,-51.274027,-0.881077,0.000000,11.515,normal
1,1,0,0.073174,1.981245,-0.629966,1,CC,E,35-N,Video_001,...,0.763748,0.007194,0.138270,-0.881077,-15.792735,60.562228,-2.033736,126.445970,11.515,normal
2,2,0,0.123811,1.972938,-0.639559,1,CC,E,35-N,Video_001,...,1.150466,0.015596,0.088876,-1.812237,-8.807270,142.715592,-2.177735,147.635591,11.515,normal
3,3,0,0.181472,1.974790,-0.639965,1,CC,E,35-N,Video_001,...,1.312186,0.015662,0.027576,-1.690928,-0.265367,107.160874,-1.927719,764.513321,11.515,normal
4,4,0,0.224938,1.972984,-0.641756,1,CC,E,35-N,Video_001,...,1.431348,0.017727,0.071411,-1.780087,3.049804,56.515627,-0.930846,305.291105,11.515,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44319,44319,320,11.291740,2.965504,-3.120556,3,LA,R,35,Video_006,...,0.245230,1.719952,0.126393,-2.648078,0.006991,0.096506,-2.552953,16.924950,11.515,normal
44320,44320,320,11.348250,2.959012,-3.125446,3,LA,R,35,Video_006,...,0.681606,1.727987,0.101667,-2.647365,0.021723,0.140003,-2.293583,78.380508,11.515,normal
44321,44321,320,11.393450,2.957334,-3.128929,3,LA,R,35,Video_006,...,1.167128,1.731117,0.169209,-2.646054,0.023688,-0.067894,-2.407994,75.171379,11.515,normal
44322,44322,320,11.429330,2.950067,-3.134238,3,LA,R,35,Video_006,...,0.186546,1.740035,0.201462,-2.645356,0.018080,-0.110120,-2.490801,11.765173,11.515,normal


In [3]:
columnas = list(df_resumen.select_dtypes(include='float64').columns)
columnas.remove('duracion_video')
columnas

['r_promedio',
 'r_mediana',
 'r_centrado_promedio',
 'r_centrado_mediana',
 'v_promedio',
 'v_mediana',
 'a_promedio',
 'a_mediana',
 'dr_promedio',
 'dr_mediana',
 'dr_centrado_promedio',
 'dr_centrado_mediana',
 'theta_r_promedio',
 'theta_r_mediana',
 'theta_r_centrado_promedio',
 'theta_r_centrado_mediana',
 'v_angular_promedio',
 'v_angular_mediana',
 'v_angular_centrado_promedio',
 'v_angular_centrado_mediana',
 'a_angular_promedio',
 'a_angular_mediana',
 'a_angular_centrado_promedio',
 'a_angular_centrado_mediana',
 'curvatura_promedio',
 'curvatura_mediana',
 'theta_v_promedio',
 'theta_v_mediana']

In [4]:
cc = df_resumen[(df_resumen['especie'] == 'CC') & (df_resumen['tratamiento'] == 'E')] 
la = df_resumen[(df_resumen['especie'] == 'LA') & (df_resumen['tratamiento'] == 'E')]

In [5]:
print(cc['nivel_salinidad'].unique())
print(la['nivel_salinidad'].unique())

['normal' 'alta' 'baja']
['normal' 'alta' 'baja']


In [6]:
grupos = {'CC': cc, 'LA': la}
resultados = {}

factor = 'nivel_salinidad'

for nombre_grupo, df in grupos.items():
    clave = f"{nombre_grupo}"
    resultados[clave] = pd.concat([testear_supuestos(df_resumen, var, factor) for var in columnas],
                                  ignore_index=True)
    

df_resultados = pd.concat([pd.DataFrame(lista).assign(grupo=clave) for clave, lista in resultados.items()],
                          ignore_index=True)

## LOCOS

### Análisis de la varianza

In [7]:
resultados_cc = df_resultados[(df_resultados['grupo'] == 'CC')]

variables_resumen_por_test = (resultados_cc[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("LOCOS - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
#filtro
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_cc.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

#ver ranking
df_efectos_cc = analizar_grupo(cc, listas_por_test, 'nivel_salinidad')
pvalues_significativos_cc = df_efectos_cc.loc[(df_efectos_cc['p_value'] < 0.05)]
variables_candidatas_cc = pvalues_significativos_cc['variable'].tolist()
variables_candidatas_cc = set(variables_candidatas_cc) #podemos mostrar el dataframe completo también
print(pvalues_significativos_cc)

LOCOS - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_v_mediana']
Puedo aplicar Kruskal-Wallis en: ['r_promedio', 'r_mediana', 'r_centrado_promedio', 'r_centrado_mediana', 'v_promedio', 'v_mediana', 'a_promedio', 'a_mediana', 'dr_promedio', 'dr_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana', 'theta_r_promedio', 'theta_r_mediana', 'theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'v_angular_promedio', 'v_angular_mediana', 'v_angular_centrado_promedio', 'v_angular_centrado_mediana', 'a_angular_promedio', 'a_angular_mediana', 'a_angular_centrado_promedio', 'a_angular_centrado_mediana', 'curvatura_promedio', 'curvatura_mediana', 'theta_v_promedio']
            variable            test       p_value        η²     ω²/ε²
1          r_mediana  Kruskal-Wallis  4.748453e-07  0.549444  0.531776
0         r_promedio  Kruskal-Wallis  4.918968e-07  0.548113  0.530392
12  theta_r_promedio  Kruskal-Wallis  2.035033e-03  0.233858  0.203813
13   theta_r_mediana  Kruskal-Wal

### Pruebas post hoc

In [8]:
variables_tukey_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == 'ANOVA') &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

variables_gh_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == "Welch's ANOVA") &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

variables_dunn_cc = list(df_efectos_cc.loc[
    (df_efectos_cc['test'] == 'Kruskal-Wallis') &
    (df_efectos_cc['variable'].isin(variables_candidatas_cc)),
    'variable'])

In [9]:
if variables_tukey_cc:
    tukey_cc = posthoc_anova_tukey(cc, variables_tukey_cc, factor)

if variables_gh_cc:
    gh_cc = posthoc_welch_gameshowell(cc, variables_gh_cc, factor)
    
if variables_dunn_cc:
    dunn_cc = posthoc_kruskal_dunn(cc, variables_dunn_cc, factor)

In [10]:
if 'tukey_cc' in locals():
    posthocs_tukey_cc = {variable: tukey_cc.loc[tukey_cc['variable'] == variable] for variable in variables_tukey_cc}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_cc' in locals():
    posthocs_gh_cc = {variable: gh_cc.loc[gh_cc['variable'] == variable] for variable in variables_gh_cc}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_cc' in locals():
    posthocs_dunn_cc = {variable: dunn_cc.loc[dunn_cc['variable'] == variable] for variable in variables_dunn_cc}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Dunn. Revisar


In [11]:
posthocs_dunn_cc

{'r_mediana':    grupo1  grupo2       p-valor   variable
 0    alta    alta  1.000000e+00  r_mediana
 1    baja    alta  2.098234e-07  r_mediana
 2  normal    alta  3.060081e-02  r_mediana
 3    alta    baja  2.098234e-07  r_mediana
 4    baja    baja  1.000000e+00  r_mediana
 5  normal    baja  3.728370e-03  r_mediana
 6    alta  normal  3.060081e-02  r_mediana
 7    baja  normal  3.728370e-03  r_mediana
 8  normal  normal  1.000000e+00  r_mediana,
 'r_promedio':     grupo1  grupo2       p-valor    variable
 9     alta    alta  1.000000e+00  r_promedio
 10    baja    alta  2.249765e-07  r_promedio
 11  normal    alta  3.974863e-02  r_promedio
 12    alta    baja  2.249765e-07  r_promedio
 13    baja    baja  1.000000e+00  r_promedio
 14  normal    baja  2.765024e-03  r_promedio
 15    alta  normal  3.974863e-02  r_promedio
 16    baja  normal  2.765024e-03  r_promedio
 17  normal  normal  1.000000e+00  r_promedio,
 'theta_r_promedio':     grupo1  grupo2   p-valor          variable
 18

## ERIZOS

### Análisis de la varianza

In [12]:
resultados_la = df_resultados[(df_resultados['grupo'] == 'LA')]

variables_resumen_por_test = (resultados_la[['variable', 'test_sugerido']].drop_duplicates()
    .groupby('test_sugerido')['variable']
    .unique()
    .to_dict())

print("ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS")
for tipo, lista in variables_resumen_por_test.items():
    print(f"Puedo aplicar {tipo} en: {list(lista)}")
    
#filtro
listas_por_test = {
    tipo: grupo['variable'].drop_duplicates().tolist()
    for tipo, grupo in resultados_la.dropna(subset=['test_sugerido']).groupby('test_sugerido')}

#ver ranking
df_efectos_la = analizar_grupo(la, listas_por_test, 'nivel_salinidad')
pvalues_significativos_la = df_efectos_la.loc[(df_efectos_la['p_value'] < 0.05)]
variables_candidatas_la = pvalues_significativos_la['variable'].tolist()
variables_candidatas_la = set(variables_candidatas_la) #podemos mostrar el dataframe completo también
print(pvalues_significativos_la)

ERIZOS - QUÉ APLICAR SEGÚN SUPUESTOS
Puedo aplicar ANOVA en: ['theta_v_mediana']
Puedo aplicar Kruskal-Wallis en: ['r_promedio', 'r_mediana', 'r_centrado_promedio', 'r_centrado_mediana', 'v_promedio', 'v_mediana', 'a_promedio', 'a_mediana', 'dr_promedio', 'dr_mediana', 'dr_centrado_promedio', 'dr_centrado_mediana', 'theta_r_promedio', 'theta_r_mediana', 'theta_r_centrado_promedio', 'theta_r_centrado_mediana', 'v_angular_promedio', 'v_angular_mediana', 'v_angular_centrado_promedio', 'v_angular_centrado_mediana', 'a_angular_promedio', 'a_angular_mediana', 'a_angular_centrado_promedio', 'a_angular_centrado_mediana', 'curvatura_promedio', 'curvatura_mediana', 'theta_v_promedio']
                      variable            test   p_value        η²     ω²/ε²
0                   r_promedio  Kruskal-Wallis  0.000184  0.390975  0.361974
1                    r_mediana  Kruskal-Wallis  0.000193  0.388660  0.359549
17           v_angular_mediana  Kruskal-Wallis  0.006679  0.227675  0.190897
16      

### Pruebas post hoc

In [13]:
variables_tukey_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == 'ANOVA') &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

variables_gh_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == "Welch's ANOVA") &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

variables_dunn_la = list(df_efectos_la.loc[
    (df_efectos_la['test'] == 'Kruskal-Wallis') &
    (df_efectos_la['variable'].isin(variables_candidatas_la)),
    'variable'])

In [14]:
if variables_tukey_la:
    tukey_la = posthoc_anova_tukey(la, variables_tukey_la, factor)

if variables_gh_la:
    gh_la = posthoc_welch_gameshowell(la, variables_gh_la, factor)
    
if variables_dunn_la:
    dunn_la = posthoc_kruskal_dunn(la, variables_dunn_la, factor)

In [15]:
if 'tukey_la' in locals():
    posthocs_tukey_la = {variable: tukey_la.loc[tukey_la['variable'] == variable] for variable in variables_tukey_la}
    print("Hay df con post hoc para Tukey. Revisar")
    
if 'gh_la' in locals():
    posthocs_gh_la = {variable: gh_la.loc[gh_la['variable'] == variable] for variable in variables_gh_la}
    print("Hay df con post hoc para GH. Revisar")
    
if 'dunn_la' in locals():
    posthocs_dunn_la = {variable: dunn_la.loc[dunn_la['variable'] == variable] for variable in variables_dunn_la}
    print("Hay df con post hoc para Dunn. Revisar")

Hay df con post hoc para Dunn. Revisar


In [16]:
posthocs_dunn_la

{'r_promedio':    grupo1  grupo2   p-valor    variable
 0    alta    alta  1.000000  r_promedio
 1    baja    alta  0.000292  r_promedio
 2  normal    alta  0.408182  r_promedio
 3    alta    baja  0.000292  r_promedio
 4    baja    baja  1.000000  r_promedio
 5  normal    baja  0.015515  r_promedio
 6    alta  normal  0.408182  r_promedio
 7    baja  normal  0.015515  r_promedio
 8  normal  normal  1.000000  r_promedio,
 'r_mediana':     grupo1  grupo2   p-valor   variable
 9     alta    alta  1.000000  r_mediana
 10    baja    alta  0.000335  r_mediana
 11  normal    alta  0.445436  r_mediana
 12    alta    baja  0.000335  r_mediana
 13    baja    baja  1.000000  r_mediana
 14  normal    baja  0.013770  r_mediana
 15    alta  normal  0.445436  r_mediana
 16    baja  normal  0.013770  r_mediana
 17  normal  normal  1.000000  r_mediana,
 'v_angular_mediana':     grupo1  grupo2   p-valor           variable
 18    alta    alta  1.000000  v_angular_mediana
 19    baja    alta  0.014635  v